<a href="https://colab.research.google.com/github/sandeepa-ukr/AI-Based-Crop-Predection/blob/main/crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ML imports
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [4]:
crop_files = [
    ("apples.csv", "apples"),
    ("apricots.csv", "apricots"),
    ("areca_nuts.csv", "areca nuts"),
    ("bananas.csv", "bananas"),
    ("barley.csv", "barley"),
    ("beans_dry.csv", "beans dry"),
    ("beans_green.csv", "beans green"),
    ("cabbage.csv", "cabbage"),
    ("carrots.csv", "carrots"),
    ("cashewnuts.csv", "cashewnuts"),
    ("cauliflower.csv", "cauliflower"),
    ("cherries.csv", "cherries"),
    ("chilli_dry.csv", "chilli dry"),
    ("chilli_green.csv", "chilli green"),
    ("coconuts.csv", "coconuts"),
    ("coffee.csv", "coffee"),
    ("cucumber.csv", "cucumber"),
    ("eggplant.csv", "eggplant"),
    ("fennel_coriander.csv", "fennel coriander"),
    ("garlic.csv", "garlic"),
    ("ginger.csv", "ginger"),
    ("grapes.csv", "grapes"),
    ("groundnuts.csv", "groundnuts"),
    ("lemons.csv", "lemons"),
    ("lentils.csv", "lentils"),
    ("lettuce.csv", "lettuce"),
    ("maize.csv", "maize"),
    ("mangoes.csv", "mangoes"),
    ("melons.csv", "melons"),
    ("nutmeg.csv", "nutmeg"),
    ("okra.csv", "okra"),
    ("onions.csv", "onions"),
    ("oranges.csv", "oranges"),
    ("papaya.csv", "papaya"),
    ("peaches.csv", "peaches"),
    ("pears.csv", "pears"),
    ("peas.csv", "peas"),
    ("pepper.csv", "pepper"),
    ("pineapples.csv", "pineapples"),
    ("potatoes.csv", "potatoes"),
    ("pumpkins.csv", "pumpkins"),
    ("rice.csv", "rice"),
    ("safflower.csv", "safflower"),
    ("sesame.csv", "sesame"),
    ("soyabean.csv", "soyabean"),
    ("sugarcane.csv", "sugarcane"),
    ("tomatoes.csv", "tomatoes"),
    ("watermelon.csv", "watermelon"),
    ("wheat.csv", "wheat"),
]

In [5]:
crop_dfs = []
for fname, crop_name in crop_files:
    tmp = pd.read_csv(fname)
    # tmp = tmp.copy()
    # tmp['Year'] = tmp['Year'].astype(int)
    # tmp['crop_type'] = crop_name
    crop_dfs.append(tmp)

crop_df = pd.concat(crop_dfs, ignore_index=True, axis=0)
print(f"Loaded crop_df shape: {crop_df.shape}")

Loaded crop_df shape: (8526, 17)


In [6]:
# Check for basic data quality
print("\n=== Data Quality Checks ===")
print(f"Total rows: {len(crop_df)}")
print(f"Null values per column:")
print(crop_df.isnull().sum())
print(f"\nDataFrame info:")
print(crop_df.info())

# Check unique values in key columns
print(f"\nUnique crops: {crop_df['Item'].nunique() if 'Item' in crop_df.columns else 'Item column not found'}")
print(f"Year range: {crop_df['Year'].min() if 'Year' in crop_df.columns else 'Year column not found'} - {crop_df['Year'].max() if 'Year' in crop_df.columns else 'Year column not found'}")


=== Data Quality Checks ===
Total rows: 8526
Null values per column:
Domain Code                0
Domain                     0
Area Code (FAO)            0
Area                       0
Element Code               0
Element                    0
Item Code (FAO)            0
Item                       0
Year Code                  0
Year                       0
Unit                       0
Value                      0
Flag                    3547
Flag Description           0
Nitrogen (N) kg/ha         0
Phosphorus (P) kg/ha       0
Potassium (K) kg/ha        0
dtype: int64

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8526 entries, 0 to 8525
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Domain Code           8526 non-null   object 
 1   Domain                8526 non-null   object 
 2   Area Code (FAO)       8526 non-null   int64  
 3   Area                  8526 non-null   ob

In [7]:
rainfall = pd.read_csv('rainfall.csv')
temperature = pd.read_csv('temperature.csv')

In [8]:
for dfc in (rainfall, temperature):
    if 'YEAR' in dfc.columns:
        dfc.rename(columns={'YEAR': 'Year'}, inplace=True)
    if 'Year' not in dfc.columns:
        raise ValueError("Climate files must have a 'Year' column")

In [9]:

climate_df = pd.merge(rainfall, temperature, on='Year', how='outer')
merged_df = pd.merge(climate_df, crop_df, on='Year', how='outer')

# Or in one line
# merged_df = pd.merge(pd.merge(df1, df2, on='Year', how='outer'), df3, on='Year', how='outer')

In [10]:
# compute seasonal aggregates if not present
if 'winter_avg' not in climate_df.columns:
    # winter defined here as DEC (prev), JAN, FEB — approximation using available DEC/JAN/FEB
    # Because DEC previous-year isn't directly available per-row, we compute seasonal proxies as mean of DJF where available
    climate_df['winter_avg'] = climate_df[['DEC', 'JAN', 'FEB']].mean(axis=1)
if 'summer_avg' not in climate_df.columns:
    climate_df['summer_avg'] = climate_df[['MAR', 'APR', 'MAY']].mean(axis=1)
if 'monsoon_avg' not in climate_df.columns:
    climate_df['monsoon_avg'] = climate_df[['JUN', 'JUL', 'AUG', 'SEP']].mean(axis=1)

climate_df['climate_std'] = climate_df[['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']].std(axis=1)

print("Climate preprocessing done.")

Climate preprocessing done.


In [11]:
climate_df.drop('ANN', axis=1, inplace=True)

In [12]:
print(f"Climate data rows: {len(climate_df)}")
print(f"Crop data rows: {len(crop_df)}")
print(f"Total rows in merged_df: {len(merged_df)}")
print(f"Unclassified rows: {len(merged_df) - len(climate_df) - len(crop_df)}")

# Check what's in climate data
print("\n=== Climate Data Sample ===")
print(climate_df[['Year', 'JAN', 'FEB', 'ANNUAL']].head())
print(f"Climate data columns: {climate_df.columns.tolist()}")

# Check what's in crop data
print("\n=== Crop Data Sample ===")
print(crop_df[['Year', 'Item', 'Element', 'Value', 'Unit']].head())
print(f"crop data columns: {crop_df.columns.tolist()}")
print(f"Unique crops: {crop_df['Item'].nunique()}")
print(f"Unique elements: {crop_df['Element'].unique()}")


print(f"Combined climate and crop data columns: {merged_df.columns.tolist()}")

Climate data rows: 58
Crop data rows: 8526
Total rows in merged_df: 8530
Unclassified rows: -54

=== Climate Data Sample ===
   Year   JAN   FEB  ANNUAL
0  1961  26.1  34.8   24.00
1  1962  12.6  21.6   24.04
2  1963   6.8   9.8   24.15
3  1964  18.6  14.1   24.10
4  1965  11.8  28.1   24.07
Climate data columns: ['Year', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'ANNUAL', 'JAN-FEB', 'MAR-MAY', 'JUN-SEP', 'OCT-DEC', 'winter_avg', 'summer_avg', 'monsoon_avg', 'climate_std']

=== Crop Data Sample ===
   Year    Item         Element  Value Unit
0  1965  Apples  Area harvested  44500   ha
1  1966  Apples  Area harvested  38500   ha
2  1967  Apples  Area harvested  33700   ha
3  1968  Apples  Area harvested  26500   ha
4  1969  Apples  Area harvested  21700   ha
crop data columns: ['Domain Code', 'Domain', 'Area Code (FAO)', 'Area', 'Element Code', 'Element', 'Item Code (FAO)', 'Item', 'Year Code', 'Year',